# Different Ways to Remove Duplicates in Hive

Removing duplicate records is a common task in Hive, especially in data warehousing and ETL pipelines. Since Hive tables are immutable (without ACID operations), deduplication is typically done by creating a new table or overwriting an existing one.

---

# 1. Using `SELECT DISTINCT`

## Use Case
- Remove completely duplicate rows.
- All columns are considered.

### Example

```sql
SELECT DISTINCT *
FROM employee;
```

### Input

| id | name | age |
|----|------|-----|
| 1 | Alice | 25 |
| 1 | Alice | 25 |
| 2 | Bob | 30 |

### Output

| id | name | age |
|----|------|-----|
| 1 | Alice | 25 |
| 2 | Bob | 30 |

### Advantages
- Simple and easy to use.
- Best for removing exact duplicate rows.

### Limitations
- Cannot specify which duplicate record to keep.
- Expensive for large datasets.

---

# 2. Using `GROUP BY`

## Use Case
- Remove duplicates based on one or more columns.
- Perform aggregation while deduplicating.

### Example

```sql
SELECT
    id,
    MAX(name) AS name,
    MAX(age) AS age
FROM employee
GROUP BY id;
```

### Advantages
- Supports aggregations.
- Useful for reporting and summarization.

### Limitations
- Requires aggregation on non-grouped columns.

---

# 3. Using `ROW_NUMBER()` Window Function (Most Recommended)

## Use Case
- Keep the latest, earliest, or highest-priority record among duplicates.

### Example

```sql
SELECT id,
       name,
       age,
       updated_date
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY updated_date DESC
           ) AS rn
    FROM employee
) t
WHERE rn = 1;
```

### Advantages
- Deterministically selects the record to keep.
- Best practice for production ETL.

---

# 4. Using `RANK()` Window Function

## Use Case
- Keep all rows with the highest-ranked value.

### Example

```sql
SELECT *
FROM (
    SELECT *,
           RANK() OVER (
               PARTITION BY id
               ORDER BY salary DESC
           ) AS rnk
    FROM employee
) t
WHERE rnk = 1;
```

### Advantages
- Keeps all tied records.
- Useful when multiple rows share the same highest value.

---

# 5. Using `DENSE_RANK()`

## Use Case
- Similar to `RANK()`, but without gaps in ranking.

### Example

```sql
SELECT *
FROM (
    SELECT *,
           DENSE_RANK() OVER (
               PARTITION BY id
               ORDER BY salary DESC
           ) AS drnk
    FROM employee
) t
WHERE drnk = 1;
```

### Advantages
- Useful for handling ties while maintaining continuous ranking.

---

# 6. Using `INSERT OVERWRITE` with Deduplication

## Use Case
- Permanently remove duplicates from a Hive table.

### Example

```sql
INSERT OVERWRITE TABLE employee_clean

SELECT DISTINCT *
FROM employee;
```

Or using a window function:

```sql
INSERT OVERWRITE TABLE employee_clean

SELECT id,
       name,
       age
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY updated_date DESC
           ) AS rn
    FROM employee
) t
WHERE rn = 1;
```

### Advantages
- Common ETL pattern.
- Creates a clean deduplicated table.

---

# 7. Using Common Table Expression (CTE)

## Use Case
- Improve readability of complex deduplication queries.

### Example

```sql
WITH dedup AS (

    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY updated_date DESC
           ) AS rn
    FROM employee

)

SELECT *
FROM dedup
WHERE rn = 1;
```

### Advantages
- Makes SQL easier to read and maintain.
- Preferred for complex ETL workflows.

---

# 8. Using `COLLECT_LIST()` or `COLLECT_SET()`

## Use Case
- Aggregate duplicate values into arrays instead of removing them.

### Example

```sql
SELECT
    id,
    COLLECT_SET(name) AS names
FROM employee
GROUP BY id;
```

### Advantages
- Eliminates duplicate values within each group.
- Useful for reporting and analytics.

---

# Comparison

| Method | Removes Exact Duplicates | Removes Based on Columns | Control Over Record Selection | Production Use |
|---------|--------------------------|--------------------------|-------------------------------|----------------|
| `SELECT DISTINCT` | ✅ | ❌ | ❌ | ⭐⭐⭐ |
| `GROUP BY` | ✅ | ✅ | Aggregation only | ⭐⭐⭐⭐ |
| `ROW_NUMBER()` | ✅ | ✅ | ✅ Excellent | ⭐⭐⭐⭐⭐ |
| `RANK()` | ✅ | ✅ | Keeps ties | ⭐⭐⭐⭐ |
| `DENSE_RANK()` | ✅ | ✅ | Keeps ties | ⭐⭐⭐⭐ |
| `INSERT OVERWRITE` | Depends on query | Depends on query | Depends on logic | ⭐⭐⭐⭐⭐ |
| CTE + `ROW_NUMBER()` | ✅ | ✅ | ✅ Excellent | ⭐⭐⭐⭐⭐ |
| `COLLECT_SET()` | Removes duplicate values within groups | ✅ | N/A | ⭐⭐⭐ |

---

# Best Practices

- ✅ Use **`SELECT DISTINCT`** for small datasets with exact duplicate rows.
- ✅ Use **`GROUP BY`** when deduplication is combined with aggregation.
- ✅ Use **`ROW_NUMBER()`** to deterministically keep the latest, earliest, or highest-priority record. This is the preferred approach for production ETL.
- ✅ Use **CTEs** to make complex deduplication queries easier to read and maintain.
- ✅ Use **`INSERT OVERWRITE`** to write deduplicated data into a clean Hive table.
- ✅ Use **`COLLECT_SET()`** when you need unique values aggregated into an array rather than removing rows.

> **Interview Tip:** In Hive, the most common production approach is **`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`** combined with **`INSERT OVERWRITE`** to create or refresh a deduplicated table. This gives deterministic results and scales well for large datasets.

# 7 Different Ways to Remove Duplicates in Spark & PySpark

Removing duplicate records is one of the most common data cleansing operations in Spark. Depending on the requirement, Spark provides multiple ways to identify and remove duplicates.

---

# 1. Using `distinct()`

## Use Case
- Remove completely duplicate rows.
- All columns are considered.

### Example

```python
df = df.distinct()
```

### Input

| id | name | age |
|----|------|-----|
| 1 | Alice | 25 |
| 1 | Alice | 25 |
| 2 | Bob | 30 |

### Output

| id | name | age |
|----|------|-----|
| 1 | Alice | 25 |
| 2 | Bob | 30 |

### Advantages
- Simple and easy to use.
- Removes duplicate rows across all columns.

### Limitation
- Cannot specify particular columns.

---

# 2. Using `dropDuplicates()`

## Use Case
- Remove duplicate rows based on all columns or selected columns.

### Remove duplicates from all columns

```python
df = df.dropDuplicates()
```

### Remove duplicates based on selected columns

```python
df = df.dropDuplicates(["id"])
```

### Input

| id | name | age |
|----|------|-----|
| 1 | Alice | 25 |
| 1 | Alice | 28 |
| 2 | Bob | 30 |

### Output

```python
df.dropDuplicates(["id"])
```

| id | name | age |
|----|------|-----|
| 1 | Alice | 25 *(or 28)* |
| 2 | Bob | 30 |

> **Note:** Spark keeps one arbitrary record for each duplicate key unless you explicitly specify which one to keep.

---

# 3. Using `groupBy()` with Aggregation

## Use Case
- Remove duplicates while performing aggregations.

### Example

```python
from pyspark.sql.functions import first

df = (
    df.groupBy("id")
      .agg(first("name").alias("name"),
           first("age").alias("age"))
)
```

### Advantages
- Allows aggregation during deduplication.
- Useful for reporting and summarization.

---

# 4. Using Window Functions (`row_number()`)

## Use Case
- Keep the latest, earliest, or highest-priority record among duplicates.

### Example

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("id").orderBy(df.updated_date.desc())

df = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn = 1")
      .drop("rn")
)
```

### Advantages
- Full control over which duplicate to retain.
- Best approach for Slowly Changing Dimensions (SCD) and CDC pipelines.

---

# 5. Using SQL with `ROW_NUMBER()`

## Use Case
- Deduplication using Spark SQL.

### Example

```python
df.createOrReplaceTempView("employees")
```

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY updated_date DESC
           ) AS rn
    FROM employees
)
WHERE rn = 1;
```

### Advantages
- Familiar for SQL developers.
- Supports complex deduplication logic.

---

# 6. Using `groupBy()` + `max()` / `min()`

## Use Case
- Keep records based on maximum or minimum values.

### Example

```python
from pyspark.sql.functions import max

df = (
    df.groupBy("id")
      .agg(max("salary").alias("salary"))
)
```

### Advantages
- Useful for finding the highest salary, latest timestamp, or lowest value.
- Performs aggregation while removing duplicates.

---

# 7. Using `exceptAll()` or `subtract()` (Comparison-Based Deduplication)

## Use Case
- Compare two DataFrames and identify duplicate or extra records.

### Example

```python
duplicates = df1.exceptAll(df2)
```

or

```python
duplicates = df1.subtract(df2)
```

### Advantages
- Useful for data validation.
- Compare source and target datasets.
- Common in ETL reconciliation processes.

> **Difference**
>
> - `subtract()` removes duplicate rows before comparison.
> - `exceptAll()` preserves duplicate occurrences.

---

# Comparison

| Method | Removes Full Row Duplicates | Removes Based on Selected Columns | Choose Which Record to Keep | Production Use |
|---------|-----------------------------|-----------------------------------|-----------------------------|----------------|
| `distinct()` | ✅ | ❌ | ❌ | ⭐⭐⭐ |
| `dropDuplicates()` | ✅ | ✅ | ❌ | ⭐⭐⭐⭐⭐ |
| `groupBy()` + `first()` | ✅ | ✅ | Limited | ⭐⭐⭐⭐ |
| Window + `row_number()` | ✅ | ✅ | ✅ | ⭐⭐⭐⭐⭐ |
| SQL `ROW_NUMBER()` | ✅ | ✅ | ✅ | ⭐⭐⭐⭐⭐ |
| `groupBy()` + `max()` / `min()` | ✅ | ✅ | Based on aggregation | ⭐⭐⭐⭐ |
| `exceptAll()` / `subtract()` | Comparison | Comparison | N/A | ⭐⭐⭐ |

---

# Best Practices

- ✅ Use **`distinct()`** when every column must be unique.
- ✅ Use **`dropDuplicates()`** for simple deduplication based on one or more columns.
- ✅ Use **Window Functions (`row_number()`)** when you need deterministic control over which duplicate record to retain.
- ✅ Use **`groupBy()` with aggregation** when deduplication is part of a reporting or aggregation workflow.
- ✅ Use **Spark SQL `ROW_NUMBER()`** if your team primarily writes SQL.
- ✅ Use **`exceptAll()` or `subtract()`** for comparing datasets during ETL validation or reconciliation.

> **Interview Tip:** While `distinct()` and `dropDuplicates()` are the most common methods, **window functions with `row_number()` are generally considered the preferred production approach** because they allow you to deterministically select which record to keep (for example, the latest record based on a timestamp).


In [0]:
# 1. Remove exact duplicates across all columns
query_distinct = """
SELECT DISTINCT *
FROM table_name
"""

# 2. Remove duplicates based on specific columns, keeping one arbitrary record
query_group_by = """
SELECT col1, MAX(col2) AS col2, MAX(col3) AS col3
FROM table_name
GROUP BY col1
"""

# 3. Deterministically keep the latest record among duplicates using ROW_NUMBER()
query_row_number = """
SELECT col1, col2, col3, updated_date
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY col1
               ORDER BY updated_date DESC
           ) AS rn
    FROM table_name
) t
WHERE rn = 1
"""

# 4. Keep all tied records with the highest value using RANK()
query_rank = """
SELECT *
FROM (
    SELECT *,
           RANK() OVER (
               PARTITION BY col1
               ORDER BY col2 DESC
           ) AS rnk
    FROM table_name
) t
WHERE rnk = 1
"""

# 5. Aggregate duplicate values into arrays using COLLECT_SET()
query_collect_set = """
SELECT col1, COLLECT_SET(col2) AS col2_set
FROM table_name
GROUP BY col1
"""

# Different Ways to Handle Duplicates in Spark, PySpark, Hive & SQL (Real-Time Projects)

In real-world data engineering projects, duplicates occur due to multiple data sources, retries, CDC (Change Data Capture), late-arriving data, file reprocessing, or application issues. The approach to handling duplicates depends on the business requirement.

This guide covers the most commonly used duplicate-handling techniques across **Spark, PySpark, Hive, and SQL**.

---

# 1. Remove Exact Duplicates

## Scenario
Every column has identical values.

### Spark / PySpark

```python
df = df.distinct()

# or

df = df.dropDuplicates()
```

### Hive / SQL

```sql
SELECT DISTINCT *
FROM employee;
```

### Real-Time Example

A CSV file is accidentally loaded twice into the data lake.

---

# 2. Remove Duplicates Based on Business Keys

## Scenario

Multiple records have the same business key but differ in other columns.

Example:

| Customer_ID | Name | Last_Update |
|------------|------|-------------|
|101|John|2025-01-10|
|101|John|2025-01-15|

Keep only one record per Customer_ID.

### Spark

```python
df = df.dropDuplicates(["customer_id"])
```

### SQL / Hive

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY customer_id
               ORDER BY last_update DESC
           ) rn
    FROM customer
) t
WHERE rn=1;
```

### Real-Time Example

Customer master data arriving from multiple CRM systems.

---

# 3. Keep Latest Record (Most Common Production Pattern)

## Scenario

Keep the latest record using timestamp.

### PySpark

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("customer_id") \
               .orderBy("updated_date")

df = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn=1")
      .drop("rn")
)
```

### Hive / SQL

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY customer_id
               ORDER BY updated_date DESC
           ) rn
    FROM customer
) t
WHERE rn=1;
```

### Real-Time Example

Customer profile updates from mobile applications.

---

# 4. Keep Earliest Record

Sometimes business wants the first record instead of latest.

### Spark

```python
window = Window.partitionBy("emp_id") \
               .orderBy("created_date")

df = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn=1")
)
```

### Example

Employee joining history.

---

# 5. Keep Highest Priority Record

Priority determines which record survives.

Example

|Source|Priority|
|-------|--------|
|CRM|1|
|SAP|2|
|Excel|3|

### Spark

```python
window = Window.partitionBy("customer_id") \
               .orderBy("priority")

df = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn=1")
)
```

### Real-Time Example

Master Data Management (MDM)

---

# 6. Aggregate Duplicate Records

Instead of removing duplicates, aggregate them.

### SQL

```sql
SELECT
customer_id,
SUM(amount)
FROM sales
GROUP BY customer_id;
```

### Spark

```python
df.groupBy("customer_id") \
  .sum("amount")
```

### Real-Time Example

Sales reporting

---

# 7. Remove Duplicates Using GROUP BY

### Hive

```sql
SELECT
id,
MAX(name),
MAX(age)
FROM employee
GROUP BY id;
```

Useful when aggregation is required.

---

# 8. Handle Duplicates Using ROW_NUMBER()

Most common production solution.

### SQL

```sql
SELECT *
FROM
(
SELECT *,
ROW_NUMBER() OVER
(
PARTITION BY id
ORDER BY updated_date DESC
) rn
FROM employee
)t
WHERE rn=1;
```

Supported by

- SQL Server
- Oracle
- PostgreSQL
- Hive
- Spark SQL
- Databricks SQL

---

# 9. Handle Duplicate Files Before Loading

Instead of removing duplicate rows, avoid loading duplicate files.

### Example

```
sales_20250101.csv

sales_20250101.csv
```

Check

- filename
- checksum
- MD5 hash
- ingestion log

before processing.

### Real-Time Example

ADF, Airflow, Azure Databricks pipelines

---

# 10. CDC (Change Data Capture) Deduplication

Incoming CDC data

|ID|Operation|Timestamp|
|--|---------|----------|
|101|INSERT|10:00|
|101|UPDATE|10:10|
|101|UPDATE|10:15|

Keep only latest change.

### Spark

```python
window = Window.partitionBy("id") \
               .orderBy(col("timestamp").desc())

df = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn=1")
)
```

### Real-Time Example

Debezium

Kafka CDC

Oracle GoldenGate

---

# 11. Merge Duplicate Records (Delta Lake)

Instead of deleting duplicates, merge new records.

```sql
MERGE INTO customer tgt

USING customer_stage src

ON tgt.customer_id = src.customer_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *
```

### Real-Time Example

Incremental ETL

Slowly Changing Dimension (SCD)

---

# 12. Streaming Deduplication

Used only in Structured Streaming.

### PySpark

```python
stream_df = stream_df \
    .dropDuplicates(["order_id"])
```

With watermark

```python
stream_df = (
stream_df
.withWatermark("event_time","10 minutes")
.dropDuplicates(["order_id"])
)
```

### Real-Time Example

Kafka

IoT

Clickstream

Fraud Detection

---

# 13. Using EXCEPT / EXCEPT ALL

Compare source and target.

```sql
SELECT *
FROM source

EXCEPT

SELECT *
FROM target;
```

Spark

```python
source.subtract(target)
```

Useful for

- Data Validation
- Reconciliation
- Migration Testing

---

# 14. Delete Duplicates Using CTE

```sql
WITH cte AS
(
SELECT *,
ROW_NUMBER() OVER
(
PARTITION BY id
ORDER BY updated_date DESC
) rn
FROM employee
)

SELECT *
FROM cte
WHERE rn=1;
```

---

# 15. Using RANK() or DENSE_RANK()

Keep all highest-ranked records.

```sql
SELECT *
FROM
(
SELECT *,
RANK() OVER
(
PARTITION BY emp_id
ORDER BY salary DESC
) rnk
FROM employee
)t
WHERE rnk=1;
```

Useful when ties should be retained.

---

# Real-Time Project Scenarios

| Scenario | Preferred Approach |
|----------|--------------------|
| Duplicate CSV files | File checksum / ingestion log |
| Duplicate database records | ROW_NUMBER() |
| Latest customer record | ROW_NUMBER() + Timestamp |
| CDC data | ROW_NUMBER() |
| Kafka Streaming | Watermark + dropDuplicates() |
| Sales aggregation | GROUP BY |
| Delta Lake | MERGE |
| Data Validation | EXCEPT / SUBTRACT |
| Exact duplicate rows | DISTINCT |
| Business key duplicates | dropDuplicates() or ROW_NUMBER() |

---

# Interview Questions

### Q1. Difference between `DISTINCT` and `dropDuplicates()`?

- `DISTINCT` removes duplicate rows considering all columns.
- `dropDuplicates()` can remove duplicates based on all columns or a subset of columns.

---

### Q2. Why is `ROW_NUMBER()` preferred in production?

Because it allows deterministic selection of which duplicate record to keep (latest, earliest, highest priority, etc.), making the deduplication logic predictable and aligned with business rules.

---

### Q3. How do you remove duplicates in streaming data?

Use:

- Watermark
- `dropDuplicates()`

```python
.withWatermark("event_time","10 minutes")
.dropDuplicates(["order_id"])
```

---

### Q4. How do you avoid duplicate file processing?

Maintain an ingestion log or metadata table that stores processed filenames, file hashes, or checksums. Skip files that have already been processed.

---

# Best Practices

- ✅ Never rely solely on `DISTINCT` for production ETL; define clear business keys.
- ✅ Prefer `ROW_NUMBER()` with an `ORDER BY` clause for deterministic deduplication.
- ✅ Use `MERGE` for incremental loads and Slowly Changing Dimensions (SCD) in Delta Lake.
- ✅ In streaming applications, combine `dropDuplicates()` with watermarks to manage state and late-arriving data.
- ✅ Prevent duplicates as early as possible by validating files, tracking ingestion metadata, and enforcing primary/business key rules.
- ✅ Choose the deduplication strategy based on business requirements (latest record, earliest record, highest priority, or aggregated values), not just technical convenience.